# 📚 Taller 4: Biblioteca Semillero — Recomendando libros con Embeddings

Bienvenidos a la **Biblioteca Semillero**, el archivo más grande y desordenado de la ciudad.

Durante 40 años, **Doña Ana**, la bibliotecaria, recomendaba libros de memoria. Un lector llegaba y decía *"quiero algo sobre aventuras en el mar"* y ella respondía sin pensarlo: *"Moby Dick, La Odisea, y si te gusta lo moderno... Vida de Pi"*.

**Doña Ana se jubiló el lunes pasado.**

Tu misión: enseñarle su oficio a una IA usando **embeddings**.

### Lo que aprenderás hoy

| Concepto | Qué hace | Equivalente bibliotecario |
|---|---|---|
| **Embedding** | Convierte texto en vector | "Entender el tema del libro" |
| **Similitud coseno** | Compara dos vectores | "¿Estos libros se parecen?" |
| **Búsqueda semántica** | Busca los más similares | "Recomiéndame 3 libros como este" |
| **Proyección 2D** | Visualiza vecindarios | "El mapa de la biblioteca" |

⏱️ **Tiempo estimado:** 30 minutos

> ⚠️ **Antes de empezar:** asegúrate de tener Ollama corriendo (`ollama serve` en otra terminal). La siguiente celda descargará automáticamente el modelo de embeddings que necesitamos.

In [ ]:
# 1. Instalamos las librerias de Python
!pip install ollama numpy scikit-learn matplotlib -q

# 2. Descargamos el modelo de embeddings (si ya lo tienes, Ollama lo detecta y sigue)
!ollama pull nomic-embed-text

In [ ]:
import ollama
import numpy as np

MODELO_EMBEDDING = "nomic-embed-text"

def vectorizar(texto):
    """Convierte un texto en su embedding (vector de numeros)."""
    respuesta = ollama.embeddings(model=MODELO_EMBEDDING, prompt=texto)
    return np.array(respuesta["embedding"])

def similitud_coseno(v1, v2):
    """Mide que tan parecidos son dos vectores. Rango: -1 (opuestos) a 1 (identicos)."""
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# Prueba rapida
v = vectorizar("hola mundo")
print(f"Biblioteca Semillero conectada.")
print(f"Modelo: {MODELO_EMBEDDING}")
print(f"Dimensiones del vector: {len(v)}")

## 1️⃣ El Catálogo

Doña Ana dejó una libreta con sus libros favoritos y las sinopsis que ella misma escribió. Vamos a cargarlos y convertir cada sinopsis en un vector — así la IA entendera *"de qué trata"* cada libro.

In [ ]:
# El catalogo de Dona Ana
catalogo = [
    {"titulo": "1984",
     "sinopsis": "Un hombre vive en una sociedad totalitaria donde el gobierno vigila cada pensamiento y controla la verdad."},
    {"titulo": "Un Mundo Feliz",
     "sinopsis": "Sociedad futurista donde la gente es feliz gracias a drogas y condicionamiento, pero sin libertad real."},
    {"titulo": "Orgullo y Prejuicio",
     "sinopsis": "En la Inglaterra del siglo XIX, una joven inteligente se enamora de un hombre rico y orgulloso tras varios malentendidos."},
    {"titulo": "Cien Anos de Soledad",
     "sinopsis": "La historia de la familia Buendia en un pueblo mitico llamado Macondo, con realismo magico y generaciones de tragedia."},
    {"titulo": "El Codigo Da Vinci",
     "sinopsis": "Un profesor resuelve acertijos y simbolos religiosos para descubrir un secreto oculto de la iglesia catolica."},
    {"titulo": "El Nombre de la Rosa",
     "sinopsis": "Un monje investiga asesinatos misteriosos en una abadia medieval llena de libros prohibidos."},
    {"titulo": "Moby Dick",
     "sinopsis": "El capitan Ahab persigue obsesivamente a una ballena blanca gigante a traves de los mares del mundo."},
    {"titulo": "La Odisea",
     "sinopsis": "Un heroe griego intenta regresar a casa tras la guerra enfrentando monstruos, dioses y tormentas en el mar."},
    {"titulo": "Sapiens",
     "sinopsis": "Un recorrido por la historia de la humanidad desde la prehistoria hasta la era moderna, explicando nuestras revoluciones."},
    {"titulo": "Breve Historia del Tiempo",
     "sinopsis": "Stephen Hawking explica agujeros negros, relatividad y el origen del universo para lectores sin formacion cientifica."},
]

# Vectorizamos cada libro
print("Procesando catalogo...\n")
for libro in catalogo:
    libro["vector"] = vectorizar(libro["sinopsis"])
    print(f"  [OK] {libro['titulo']}")

print(f"\n{len(catalogo)} libros catalogados. Cada uno vive en un espacio de {len(catalogo[0]['vector'])} dimensiones.")

## 2️⃣ ¿Se Parecen Estos Libros?

Antes de recomendar, verifiquemos que la IA realmente *entiende* los libros. Vamos a comparar pares — unos que Doña Ana clasificaría como **parecidos** y otros como **completamente distintos**.

Recuerda la escala de similitud coseno:

| Valor | Relación |
|---|---|
| 0.9+ | Casi identicos |
| 0.7 - 0.9 | Muy relacionados |
| 0.4 - 0.7 | Mismo tema general |
| 0.0 - 0.4 | Poco o nada relacionados |

In [ ]:
def comparar(titulo1, titulo2):
    l1 = next(l for l in catalogo if l["titulo"] == titulo1)
    l2 = next(l for l in catalogo if l["titulo"] == titulo2)
    sim = similitud_coseno(l1["vector"], l2["vector"])
    print(f"  [{sim:.3f}]  '{titulo1}'  vs  '{titulo2}'")
    return sim

print("PARES QUE DEBERIAN SER PARECIDOS:")
print("-" * 60)
comparar("1984", "Un Mundo Feliz")             # ambos distopias
comparar("Moby Dick", "La Odisea")               # ambos aventuras en el mar
comparar("Sapiens", "Breve Historia del Tiempo") # ambos divulgacion cientifica

print("\nPARES QUE DEBERIAN SER DISTINTOS:")
print("-" * 60)
comparar("1984", "Orgullo y Prejuicio")                  # distopia vs romance
comparar("Moby Dick", "El Codigo Da Vinci")              # aventura marina vs thriller religioso
comparar("La Odisea", "Breve Historia del Tiempo")       # epica antigua vs fisica moderna

## 3️⃣ El Lector Pregunta

Ahora viene lo bueno. Un lector entra a la biblioteca y dice:

> *"Quiero algo sobre una sociedad donde no hay libertad"*

La consulta **no es el título de ningún libro** y **no usa las mismas palabras** que las sinopsis. La IA tiene que entender la **intención** y sugerir los libros más cercanos en significado.

In [ ]:
def recomendar(consulta, top_k=3):
    v_consulta = vectorizar(consulta)
    puntajes = []
    for libro in catalogo:
        sim = similitud_coseno(v_consulta, libro["vector"])
        puntajes.append((sim, libro["titulo"], libro["sinopsis"]))
    puntajes.sort(reverse=True)

    print(f"Lector: \"{consulta}\"\n")
    print(f"Top {top_k} recomendaciones:")
    print("=" * 70)
    for sim, titulo, sinopsis in puntajes[:top_k]:
        print(f"\n  [{sim:.3f}]  {titulo}")
        print(f"           {sinopsis}")
    return puntajes[:top_k]

recomendar("Quiero algo sobre una sociedad donde no hay libertad")

In [ ]:
print("\n" + "~" * 70 + "\n")
recomendar("Algo de aventuras en el oceano")

print("\n" + "~" * 70 + "\n")
recomendar("Quiero aprender sobre el universo y la ciencia")

print("\n" + "~" * 70 + "\n")
recomendar("Una historia de amor con malentendidos")

## 4️⃣ El Mapa de la Biblioteca

Los vectores viven en un espacio de cientos de dimensiones (imposible de dibujar). Pero podemos **proyectarlos a 2D** con PCA y ver los *"barrios"* de la biblioteca.

Libros del mismo género deberían agruparse naturalmente. Si no se agrupan, Doña Ana no aprobaría nuestro trabajo.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

vectores = np.array([l["vector"] for l in catalogo])
titulos = [l["titulo"] for l in catalogo]

# Reduccion de ~768 dimensiones a 2
pca = PCA(n_components=2)
coords = pca.fit_transform(vectores)

# Grafico
plt.figure(figsize=(11, 7))
plt.scatter(coords[:, 0], coords[:, 1], s=150, c="steelblue", alpha=0.6, edgecolors="navy")
for i, titulo in enumerate(titulos):
    plt.annotate(titulo, (coords[i, 0], coords[i, 1]),
                 fontsize=10, fontweight="bold",
                 xytext=(8, 8), textcoords="offset points")
plt.title("Mapa 2D de la Biblioteca Semillero", fontsize=14, fontweight="bold")
plt.xlabel("Dimension 1 (PCA)")
plt.ylabel("Dimension 2 (PCA)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Observa: los libros de temas parecidos terminan cerca unos de otros,")
print("sin que nosotros le hayamos dicho al modelo cuales son esos temas.")

## 5️⃣ El Libro Mal Catalogado

Los embeddings no son infalibles. Recordemos una de las limitaciones que vimos en teoría: **dependen de la calidad del texto**.

Si a Doña Ana le das una sinopsis vaga como *"es un libro sobre cosas"*, ella te mandaría a reescribirla. A la IA también se le confunde:

In [ ]:
# Un libro con sinopsis terrible
libro_vago = {
    "titulo": "El Misterio",
    "sinopsis": "Un libro sobre cosas que pasan a ciertas personas."
}
libro_vago["vector"] = vectorizar(libro_vago["sinopsis"])

print(f"Libro agregado: '{libro_vago['titulo']}'")
print(f"Sinopsis: \"{libro_vago['sinopsis']}\"\n")
print("Similitud con cada libro del catalogo:")
print("-" * 60)
sims = []
for libro in catalogo:
    sim = similitud_coseno(libro_vago["vector"], libro["vector"])
    sims.append((sim, libro["titulo"]))
for sim, titulo in sorted(sims, reverse=True):
    print(f"  [{sim:.3f}]  {titulo}")

print("\n> Sin informacion especifica, los puntajes son mediocres y parecidos entre si.")
print("> Leccion: un embedding es tan bueno como el texto que lo origina.")

## 6️⃣ Tu Turno: Tu Propia Sección

✏️ Doña Ana te dejó las llaves. Ahora tienes tu propia sección de la biblioteca:

1. **Agrega 3 libros** (tus favoritos, inventados, de cualquier género) al catálogo
2. **Haz 3 consultas** con lenguaje natural — usando palabras que NO aparezcan en tus sinopsis
3. **Observa**: ¿la IA recomienda lo que tú hubieras recomendado?

In [ ]:
# Tus libros (completa los campos)
mis_libros = [
    {"titulo": "", "sinopsis": ""},
    {"titulo": "", "sinopsis": ""},
    {"titulo": "", "sinopsis": ""},
]

# Descomenta cuando hayas completado tus libros:
# for libro in mis_libros:
#     libro["vector"] = vectorizar(libro["sinopsis"])
#     catalogo.append(libro)
#     print(f"Anadido al catalogo: {libro['titulo']}")

# Tus consultas (descomenta despues de agregar los libros):
# recomendar("escribe tu primera consulta aqui")
# recomendar("tu segunda consulta")
# recomendar("tu tercera consulta")

### 📋 Resumen

| Concepto | Qué aprendimos |
|---|---|
| **Embedding** | Cada texto se convierte en un vector de cientos de dimensiones |
| **Similitud coseno** | Número entre -1 y 1 que mide cercania semantica |
| **Busqueda semantica** | Encontrar lo parecido sin depender de palabras exactas |
| **Vecindarios** | Libros del mismo tema se agrupan solos en el espacio vectorial |
| **Limitaciones** | Texto vago = embedding impreciso |

### Preguntas guía

- ¿Qué pasa si dos libros distintos tienen sinopsis escritas con las mismas palabras pero significan cosas opuestas?
- ¿Por qué la proyección 2D puede *"mentir"* un poco? (pista: perdimos cientos de dimensiones)
- ¿Hubo alguna consulta tuya que **no** devolvió lo que esperabas? ¿Por qué crees que pasó?

---

📚 **Biblioteca Semillero cerrada por hoy. Doña Ana estaría orgullosa.** 👩‍🏫